# T2.1 — Autoencoder retrain (hardened) for the Stage-A anomaly screen

The first AE **collapsed** (reconstruction std ≈ 0.007, corr(input,recon) ≈ 0.10 — it output a
constant mid-grey). Root cause: too little capacity (16→32 ch), only 30 epochs, no collapse
guard, and no recorded AUROC. This notebook fixes all of that:

- deeper AE (32→64→128→256, real 40×40 bottleneck) — enough to reconstruct seabed texture
- SSIM + L1 loss, 100 epochs, cosine LR, heavy sonar-safe augmentation (140 tiles is small)
- **per-epoch reconstruction-std logging + a hard collapse guard that refuses to export**
- **records tile-level AUROC** (object-free vs object crops) — T2.1's real acceptance gate
- saves loss curves; exports ONNX (opset 13, ir≤13) and registers it

Run top-to-bottom on a Colab GPU. Target: AUROC ≥ 0.80 and recon-std well above the collapse floor.

In [ ]:
# 0. Setup
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!git clone https://github.com/YOUR_ORG/samudra-prahari.git || (cd samudra-prahari && git pull)
%cd samudra-prahari
!pip install -q -e '.[models]' pytorch-msssim scikit-learn
# Place datasets (or symlink from Drive), then (re)build the leak-free re-split:
!test -d Samudra_Prahari_Dataset_v2 && python scripts/data/index_samudra_v2.py && python scripts/data/materialize_resplit.py
import yaml; CFG = yaml.safe_load(open('configs/training.yaml'))['anomaly']
print(CFG)

In [ ]:
# 1. Config
import torch
SIZE      = 640
EPOCHS    = 100
BATCH     = 16
LR        = 1e-3
CH        = (32, 64, 128, 256)   # encoder channels (real bottleneck at 40x40)
COLLAPSE_STD = 0.02              # if val recon std < this -> collapsed, do NOT export
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIN_DIR = CFG['train_images']            # _resplit/anomaly/images/train  (object-free)
VAL_DIR   = CFG['val_images']              # _resplit/anomaly/images/val
NORMAL_TEST_GLOB = '_resplit/anomaly/images/test/*.jpg'     # object-free negatives (AUROC)
OBJECT_TEST_GLOB = '_resplit/detection/images/test/*.jpg'   # object positives (AUROC)
print('device:', DEVICE)

In [ ]:
# 2. Dataset + sonar-safe augmentation (seabed is orientation-agnostic -> flips/rot90 safe)
import glob, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader

def load_gray01(path, size=SIZE):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
    return (g.astype(np.float32) / 255.0)

class SeabedTiles(Dataset):
    def __init__(self, folder, augment):
        self.paths = sorted(glob.glob(folder + '/*.jpg')) + sorted(glob.glob(folder + '/*.png'))
        self.augment = augment
        assert self.paths, f'no images in {folder}'
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = load_gray01(self.paths[i])
        if self.augment:
            if np.random.rand() < 0.5: x = x[:, ::-1]
            if np.random.rand() < 0.5: x = x[::-1, :]
            x = np.rot90(x, k=np.random.randint(4))
            x = np.clip(x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.03, 0.03), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x))[None]  # [1,H,W]

train_dl = DataLoader(SeabedTiles(TRAIN_DIR, augment=True),  batch_size=BATCH, shuffle=True,  drop_last=True)
val_dl   = DataLoader(SeabedTiles(VAL_DIR,   augment=False), batch_size=BATCH, shuffle=False)
print('train tiles:', len(train_dl.dataset), '| val tiles:', len(val_dl.dataset))

In [ ]:
# 3. Model — convolutional autoencoder with a real bottleneck (no more collapse)
import torch.nn as nn

class ConvAE(nn.Module):
    def __init__(self, ch=CH):
        super().__init__()
        def enc(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        def dec(i, o, last=False):
            L = [nn.ConvTranspose2d(i, o, 4, 2, 1)]
            L += [nn.Sigmoid()] if last else [nn.BatchNorm2d(o), nn.ReLU(True)]
            return nn.Sequential(*L)
        c1, c2, c3, c4 = ch
        self.enc = nn.Sequential(enc(1, c1), enc(c1, c2), enc(c2, c3), enc(c3, c4))   # 640->40
        self.dec = nn.Sequential(dec(c4, c3), dec(c3, c2), dec(c2, c1), dec(c1, 1, last=True))  # 40->640
    def forward(self, x): return self.dec(self.enc(x))

model = ConvAE().to(DEVICE)
print(sum(p.numel() for p in model.parameters())/1e6, 'M params')

In [ ]:
# 4. Loss (SSIM + L1) + training with per-epoch recon-std logging
import torch.nn.functional as F
from pytorch_msssim import ssim

def loss_fn(recon, x):
    l1 = F.l1_loss(recon, x)
    dssim = 1 - ssim(recon, x, data_range=1.0, size_average=True)
    return 0.5 * dssim + 0.5 * l1

opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

@torch.no_grad()
def val_stats():
    model.eval(); losses, stds = [], []
    for xb in val_dl:
        xb = xb.to(DEVICE); rec = model(xb)
        losses.append(loss_fn(rec, xb).item())
        stds.append(rec.std(dim=(2, 3)).mean().item())   # spatial std -> ~0 means collapsed
    return float(np.mean(losses)), float(np.mean(stds))

hist = {'train': [], 'val': [], 'recon_std': []}
for ep in range(EPOCHS):
    model.train(); tl = []
    for xb in train_dl:
        xb = xb.to(DEVICE); opt.zero_grad()
        loss = loss_fn(model(xb), xb); loss.backward(); opt.step()
        tl.append(loss.item())
    sched.step()
    vl, vstd = val_stats()
    hist['train'].append(float(np.mean(tl))); hist['val'].append(vl); hist['recon_std'].append(vstd)
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:3d}  train {np.mean(tl):.4f}  val {vl:.4f}  recon_std {vstd:.4f}'
              + ('   <-- COLLAPSE WARNING' if vstd < COLLAPSE_STD else ''))

In [ ]:
# 5. Loss curves + collapse guard
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from pathlib import Path
Path('out').mkdir(exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['train'], label='train'); ax[0].plot(hist['val'], label='val')
ax[0].set_title('SSIM+L1 loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].axhline(COLLAPSE_STD, color='r', ls='--', label='collapse floor')
ax[1].plot(hist['recon_std'], label='recon std'); ax[1].set_title('reconstruction std'); ax[1].legend()
fig.savefig('out/t21_loss.png', dpi=110); print('saved out/t21_loss.png')

final_std = hist['recon_std'][-1]
assert final_std >= COLLAPSE_STD, (
    f'COLLAPSED: recon_std {final_std:.4f} < {COLLAPSE_STD}. Do NOT export. '
    'Increase capacity/epochs or lower LR and retrain.')
print(f'OK — recon_std {final_std:.4f} (not collapsed)')

In [ ]:
# 6. Tile-level AUROC — object-free (negatives) vs object crops (positives). T2.1's real gate.
import glob
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def anomaly_score(path):
    x = torch.from_numpy(load_gray01(path))[None, None].to(DEVICE)
    rec = model(x)
    return torch.abs(x - rec).mean().item()   # mean reconstruction error

model.eval()
neg = [anomaly_score(p) for p in glob.glob(NORMAL_TEST_GLOB)]        # object-free -> low
pos = [anomaly_score(p) for p in glob.glob(OBJECT_TEST_GLOB)]        # objects     -> high
auroc = roc_auc_score([0]*len(neg) + [1]*len(pos), neg + pos)
print(f'tile AUROC (object vs no-object) = {auroc:.3f}  (target >= 0.80)  '
      f'neg={len(neg)} pos={len(pos)}  mean neg {np.mean(neg):.3f} vs pos {np.mean(pos):.3f}')

In [ ]:
# 7. Export ONNX (opset 13, ir<=13 for onnxruntime) + register with metrics
import json, onnx, torch
model.eval().cpu()
dummy = torch.randn(1, 1, SIZE, SIZE)
torch.onnx.export(model, dummy, 'models/stage_a_ae.onnx', opset_version=13,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'b'}, 'output': {0: 'b'}})
m = onnx.load('models/stage_a_ae.onnx')
if m.ir_version > 10:            # some onnx builds emit ir 14; onnxruntime here supports <=13
    m.ir_version = 10; onnx.save(m, 'models/stage_a_ae.onnx')

json.dump({'task': 'T2.1', 'test_loss': round(hist['val'][-1], 4), 'epochs': EPOCHS,
           'recon_std': round(final_std, 4), 'auroc': round(float(auroc), 4)},
          open('out/metrics_t21.json', 'w'), indent=2)

!python scripts/register_model.py --onnx models/stage_a_ae.onnx --name stage_a_ae \
    --trained-on 'AnomalyScreen normal_seabed (synthetic, object-free)' --metrics out/metrics_t21.json
# copy weights back to Drive
!cp models/stage_a_ae.onnx* /content/drive/MyDrive/ 2>/dev/null; echo done

## After this runs
1. Confirm the printout: **recon_std well above 0.02** and **AUROC ≥ 0.80**.
2. Copy `models/stage_a_ae.onnx` (+ `.data` if present), `models/registry.json`, and
   `out/metrics_t21.json` back into the repo.
3. Locally re-run T2.2: `python scripts/eval/stage_a_recall.py --split test` — the recall
   curve will now be meaningful, and T2.2 unblocks.

If it still collapses (recon_std stays low), raise EPOCHS to 200 and/or CH to (48,96,192,384),
and lower LR to 5e-4.